In [1]:
# FULL TRAINING — paste into a NEW notebook cell and run (from C:\MedicinalPlantApp)
import json
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers
import tensorflow as tf

BASE = Path(r"C:\MedicinalPlantApp")
DATA_DIR = BASE / "data"
MODELS_DIR = BASE / "models"
LOGS_DIR = BASE / "logs"
MODELS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

IMG_SIZE = 160
BATCH = 16   # lower to 8 if you get OOM
EPOCHS = 20

# data generators
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=25,
                                   width_shift_range=0.15,
                                   height_shift_range=0.15,
                                   zoom_range=0.15,
                                   horizontal_flip=True,
                                   brightness_range=(0.8,1.2))
val_datagen = ImageDataGenerator(rescale=1./255)

train_flow = train_datagen.flow_from_directory(
    str(DATA_DIR/"train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical'
)
val_flow = val_datagen.flow_from_directory(
    str(DATA_DIR/"val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

with open(MODELS_DIR/"class_indices.json", "w") as f:
    json.dump(train_flow.class_indices, f)

num_classes = len(train_flow.class_indices)
print("Num classes:", num_classes)

# build model
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
base.trainable = False
x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs=base.input, outputs=outputs)

model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(MODELS_DIR/"plant_model_best.h5"), save_best_only=True),
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.TensorBoard(log_dir=str(LOGS_DIR), histogram_freq=1)
]

# Train
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Optional fine-tuning: unfreeze last 30 layers of base
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

ft_history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=5,
    callbacks=callbacks
)

# Save final model
model.save(MODELS_DIR/"plant_model_final.h5")
print("Saved final model:", (MODELS_DIR/"plant_model_final.h5").exists())


Found 11504 images belonging to 31 classes.
Found 2876 images belonging to 31 classes.
Num classes: 31


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 160, 160, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 80, 80, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 80, 80, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 80, 80, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 80, 80, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 80, 80, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 80, 80, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 80, 80, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 80, 80, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 80, 80, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 80, 80, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 80, 80, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 81, 81, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 40, 40, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,593,887 (9.89 MB)

 Trainable params: 335,903 (1.28 MB)

...

719/719 ━━━━━━━━━━━━━━━━━━━━ 179s 248ms/step - accuracy: 0.9534 - loss: 0.1334 - val_accuracy: 0.9823 - val_loss: 0.0530 - learning_rate: 0.0010
Epoch 10/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - accuracy: 0.9532 - loss: 0.1407

719/719 ━━━━━━━━━━━━━━━━━━━━ 243s 339ms/step - accuracy: 0.9580 - loss: 0.1241 - val_accuracy: 0.9889 - val_loss: 0.0393 - learning_rate: 0.0010
Epoch 12/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 234s 325ms/step - accuracy: 0.9619 - loss: 0.1200 - val_accuracy: 0.9809 - val_loss: 0.0578 - learning_rate: 0.0010
Epoch 13/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 194s 270ms/step - accuracy: 0.9613 - loss: 0.1143 - val_accuracy: 0.9753 - val_loss: 0.0681 - learning_rate: 0.0010
Epoch 14/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 188s 262ms/step - accuracy: 0.9601 - loss: 0.1238 - val_accuracy: 0.9816 - val_loss: 0.0502 - learning_rate: 0.0010
Epoch 15/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.9704 - loss: 0.0875

719/719 ━━━━━━━━━━━━━━━━━━━━ 801s 1s/step - accuracy: 0.9708 - loss: 0.0860 - val_accuracy: 0.9882 - val_loss: 0.0351 - learning_rate: 5.0000e-04
Epoch 16/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - accuracy: 0.9746 - loss: 0.0785

719/719 ━━━━━━━━━━━━━━━━━━━━ 242s 337ms/step - accuracy: 0.9737 - loss: 0.0830 - val_accuracy: 0.9924 - val_loss: 0.0234 - learning_rate: 5.0000e-04
Epoch 17/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 298s 414ms/step - accuracy: 0.9747 - loss: 0.0752 - val_accuracy: 0.9861 - val_loss: 0.0346 - learning_rate: 5.0000e-04
Epoch 18/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 297s 413ms/step - accuracy: 0.9743 - loss: 0.0762 - val_accuracy: 0.9826 - val_loss: 0.0484 - learning_rate: 5.0000e-04
Epoch 19/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 2118s 3s/step - accuracy: 0.9758 - loss: 0.0716 - val_accuracy: 0.9878 - val_loss: 0.0310 - learning_rate: 5.0000e-04
Epoch 20/20
719/719 ━━━━━━━━━━━━━━━━━━━━ 298s 414ms/step - accuracy: 0.9786 - loss: 0.0641 - val_accuracy: 0.9913 - val_loss: 0.0238 - learning_rate: 2.5000e-04
Epoch 1/5
719/719 ━━━━━━━━━━━━━━━━━━━━ 363s 483ms/step - accuracy: 0.7616 - loss: 1.1188 - val_accuracy: 0.9757 - val_loss: 0.0791 - learning_rate: 1.0000e-05
Epoch 2/5
719/719 ━━━━━━━━━━━━━━━━━━━━ 324s 451ms/

Saved final model: True


In [1]:
from pathlib import Path
BASE = Path(r"C:\MedicinalPlantApp")
print("plant_model_best.h5:", (BASE/"models"/"plant_model_best.h5").exists())
print("plant_model_final.h5:", (BASE/"models"/"plant_model_final.h5").exists())
print("class_indices.json:", (BASE/"models"/"class_indices.json").exists())


plant_model_best.h5: True
plant_model_final.h5: True
class_indices.json: True


In [2]:
# Step 2 — Evaluate final model on full validation set and show a classification report
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

BASE = Path(r"C:\MedicinalPlantApp")
MODEL_PATH = BASE/"models"/"plant_model_final.h5"
CLA_JSON = BASE/"models"/"class_indices.json"

# load model + mapping
model = tf.keras.models.load_model(str(MODEL_PATH))
with open(CLA_JSON) as f:
    class_indices = json.load(f)
idx_to_class = {v:k for k,v in class_indices.items()}
classes = [idx_to_class[i] for i in range(len(idx_to_class))]

IMG_SIZE = 160
BATCH = 16

val_datagen = ImageDataGenerator(rescale=1./255)
val_flow = val_datagen.flow_from_directory(
    str(BASE/"data"/"val"),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

# Evaluate (loss & accuracy)
loss, acc = model.evaluate(val_flow, verbose=1)
print(f"\nValidation loss: {loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")

# Predict all validation images to create classification report
val_flow.reset()
preds = model.predict(val_flow, verbose=1)
y_pred = preds.argmax(axis=1)
y_true = val_flow.classes

print("\n--- Classification report (precision, recall, f1) ---")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

# Optional: confusion matrix (counts)
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix shape:", cm.shape)
# Print a compact view: for each class, show true_count / correct_predictions
for i, cname in enumerate(classes):
    total = cm[i].sum()
    correct = cm[i,i]
    print(f"{cname}: {correct}/{total} correct ({correct/total:.3%})")



Found 2876 images belonging to 31 classes.


C:\Users\shrey\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


180/180 ━━━━━━━━━━━━━━━━━━━━ 27s 142ms/step - accuracy: 0.9757 - loss: 0.0791

Validation loss: 0.0791
Validation accuracy: 0.9757
180/180 ━━━━━━━━━━━━━━━━━━━━ 29s 154ms/step

--- Classification report (precision, recall, f1) ---


ValueError: Number of classes, 30, does not match size of target_names, 31. Try specifying the labels parameter

In [3]:
# Fixed evaluation + classification report (build class names from val_flow)
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

BASE = Path(r"C:\MedicinalPlantApp")
MODEL_PATH = BASE/"models"/"plant_model_final.h5"

# load model
model = tf.keras.models.load_model(str(MODEL_PATH))

IMG_SIZE = 160
BATCH = 16

val_datagen = ImageDataGenerator(rescale=1./255)
val_flow = val_datagen.flow_from_directory(
    str(BASE/"data"/"val"),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

# Build class name list from the generator (this will match exactly)
# val_flow.class_indices maps class_name -> index
val_class_indices = val_flow.class_indices
# Create list of class names sorted by index (0..n-1)
classes = [None] * len(val_class_indices)
for name, idx in val_class_indices.items():
    classes[idx] = name

# Evaluate (loss & accuracy)
loss, acc = model.evaluate(val_flow, verbose=1)
print(f"\nValidation loss: {loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")

# Predict and classification report
val_flow.reset()
preds = model.predict(val_flow, verbose=1)
y_pred = preds.argmax(axis=1)
y_true = val_flow.classes

print("\n--- Classification report (precision, recall, f1) ---")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

# Confusion matrix summary
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix shape:", cm.shape)
for i, cname in enumerate(classes):
    total = int(cm[i].sum())
    correct = int(cm[i,i])
    pct = (correct/total*100) if total>0 else 0.0
    print(f"{cname}: {correct}/{total} correct ({pct:.2f}%)")


Found 2876 images belonging to 31 classes.


C:\Users\shrey\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


180/180 ━━━━━━━━━━━━━━━━━━━━ 40s 198ms/step - accuracy: 0.9757 - loss: 0.0791

Validation loss: 0.0791
Validation accuracy: 0.9757
180/180 ━━━━━━━━━━━━━━━━━━━━ 38s 203ms/step

--- Classification report (precision, recall, f1) ---


ValueError: Number of classes, 30, does not match size of target_names, 31. Try specifying the labels parameter

In [4]:
# Debug + fixed classification report (run this cell)
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

BASE = Path(r"C:\MedicinalPlantApp")
MODEL_PATH = BASE/"models"/"plant_model_final.h5"

# load model
model = tf.keras.models.load_model(str(MODEL_PATH))

IMG_SIZE = 160
BATCH = 16

val_datagen = ImageDataGenerator(rescale=1./255)
val_flow = val_datagen.flow_from_directory(
    str(BASE/"data"/"val"),
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

print("\nval_flow.class_indices (class_name -> index):")
print(val_flow.class_indices)

# Build classes list using only indices present in the generator
# (this ensures ordering exactly matches labels used by y_true)
idx_to_name = {v:k for k,v in val_flow.class_indices.items()}
sorted_indices = sorted(idx_to_name.keys())
classes_ordered = [idx_to_name[i] for i in sorted_indices]
print(f"\nNumber of class indices in generator: {len(sorted_indices)}")
print("Ordered class names (by index):")
for i,name in zip(sorted_indices, classes_ordered):
    print(i, name)

# Evaluate
loss, acc = model.evaluate(val_flow, verbose=1)
print(f"\nValidation loss: {loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")

# Predict all validation images to create classification report
val_flow.reset()
preds = model.predict(val_flow, verbose=1)
y_pred = preds.argmax(axis=1)
y_true = val_flow.classes

print("\nUnique labels in y_true (sorted):", sorted(set(y_true)))
print("Max label in y_true:", int(np.max(y_true)))
print("Min label in y_true:", int(np.min(y_true)))
print("Number of unique labels in y_true:", len(set(y_true)))
print("Total val images:", len(y_true))

# Build labels and target_names that definitely match y_true:
labels = sorted(set(y_true))
target_names = [idx_to_name[i] if i in idx_to_name else f"cls_{i}" for i in labels]

print("\nUsing labels (len={}):".format(len(labels)), labels)
print("Using target_names (len={}):".format(len(target_names)), target_names[:10], " ...")

# Now safe classification report with explicit labels:
print("\n--- Classification report (precision, recall, f1) ---")
print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=4))

# Confusion matrix summary
cm = confusion_matrix(y_true, y_pred, labels=labels)
print("\nConfusion matrix shape:", cm.shape)
for i, idx in enumerate(labels):
    cname = target_names[i]
    total = int(cm[i].sum())
    correct = int(cm[i,i])
    pct = (correct/total*100) if total>0 else 0.0
    print(f"{idx} {cname}: {correct}/{total} correct ({pct:.2f}%)")


Found 2876 images belonging to 31 classes.


C:\Users\shrey\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



val_flow.class_indices (class_name -> index):
{'.ipynb_checkpoints': 0, 'Arive-Dantu': 1, 'Basale': 2, 'Betel': 3, 'Crape_Jasmine': 4, 'Curry': 5, 'Drumstick': 6, 'Fenugreek': 7, 'Guava': 8, 'Hibiscus': 9, 'Indian_Beech': 10, 'Indian_Mustard': 11, 'Jackfruit': 12, 'Jamaica_Cherry-Gasagase': 13, 'Jamun': 14, 'Jasmine': 15, 'Karanda': 16, 'Lemon': 17, 'Mango': 18, 'Mexican_Mint': 19, 'Mint': 20, 'Neem': 21, 'Oleander': 22, 'Parijata': 23, 'Peepal': 24, 'Pomegranate': 25, 'Rasna': 26, 'Rose_apple': 27, 'Roxburgh_fig': 28, 'Sandalwood': 29, 'Tulsi': 30}

Number of class indices in generator: 31
Ordered class names (by index):
0 .ipynb_checkpoints
1 Arive-Dantu
2 Basale
3 Betel
4 Crape_Jasmine
5 Curry
6 Drumstick
7 Fenugreek
8 Guava
9 Hibiscus
10 Indian_Beech
11 Indian_Mustard
12 Jackfruit
13 Jamaica_Cherry-Gasagase
14 Jamun
15 Jasmine
16 Karanda
17 Lemon
18 Mango
19 Mexican_Mint
20 Mint
21 Neem
22 Oleander
23 Parijata
24 Peepal
25 Pomegranate
26 Rasna
27 Rose_apple
28 Roxburgh_fig
29 Sand

In [1]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

plt.figure(figsize=(8,5))
plt.plot(acc, label="Training Accuracy", linewidth=2)
plt.plot(val_acc, label="Validation Accuracy", linewidth=2)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Growth Over Training")
plt.legend()
plt.grid(True)
plt.show()


NameError: name 'history' is not defined